# Exercise 5C — Groundwater Flooding and Slope Hazard Under Climate Change

**Course:** GEOV212 Hydrogeology  
**Prerequisites:** Exercise 5B (calibrated steady-state model) must be run first to produce
`data/model_input/exercise5_model_outputs.nc`.

## Learning objectives

After completing this exercise you will be able to:

1. Translate regional climate projections (precipitation and sea-level change) into
   model boundary conditions for a steady-state groundwater model.
2. Simulate the water table under current and future climate scenarios, including
   a simplified instantaneous extreme-rainfall event.
3. Map and quantify groundwater flooding hazard (water table at or near the surface).
4. Compute terrain slope from a digital elevation model and combine it with the
   modelled water table to identify areas at elevated risk of rainfall-triggered
   landslides and debris flows.
5. Communicate model results and their limitations in a concise hazard assessment.

## Scientific background

### Groundwater flooding
Groundwater flooding occurs when the water table rises to, or above, the land
surface.  In low-lying areas with shallow aquifers — common in Norwegian coastal
catchments underlain by Quaternary deposits — even a modest increase in recharge
can cause the saturated zone to intersect the ground surface, inundating
basements, roads, and agricultural land from below.  Unlike surface-water
flooding, groundwater flooding can persist for weeks or months after the rainfall
event that caused it.

### Shallow landslides and debris flows
Many Norwegian landslide and debris-flow events are triggered by intense or
prolonged rainfall that raises pore-water pressure in the shallow soil mantle.
The factor of safety against slope failure decreases as the water table rises
toward the surface.  A simplified way to identify susceptible terrain is to
combine the degree of saturation (water-table depth) with terrain slope — the
**TOPMODEL wetness index** approach.  Areas where the water table is shallow
*and* the slope is steep are flagged as elevated hazard zones.

### Climate projections for Norway
The Norwegian climate service report *Klima i Norge 2100* (Hanssen-Bauer et al.,
2015; updated 2022) projects significant increases in both mean annual
precipitation and the intensity of extreme rainfall events, particularly in
western Norway.  Sea-level rise along the Norwegian coast will further raise the
groundwater table in coastal areas by increasing the sea-boundary head.  In this
exercise you will look up the relevant projections and translate them into model
inputs.

---

## Part A — Load calibrated model outputs

We reload the NetCDF file saved at the end of Exercise 5B.  All arrays needed
for this exercise are contained in that single file.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm, BoundaryNorm
import xarray as xr
import cmcrameri.cm as cmc
from mpl_toolkits.axes_grid1 import make_axes_locatable
from affine import Affine
from pathlib import Path

import exercise_5_gw_model_utils as gwu
import exercise_5_gw_plot_utils  as gwp

# ── Load NetCDF saved by Exercise 5B ─────────────────────────────────────────
NC_PATH    = 'data/model_input/exercise5_model_outputs.nc'
MODEL_DIR  = Path('data/model_workspace_5c')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ds = xr.open_dataset(NC_PATH)

# ── Reconstruct grid arrays ───────────────────────────────────────────────────
active  = ds['active'].values.astype(bool)
dem     = ds['dem'].values.astype(float)
sw      = ds['sw'].values.astype(int)
sea     = ds['sea'].values.astype(bool)
geology = ds['geology'].values.astype(int)
rch     = ds['recharge_mm_yr'].values.astype(float) / (1000.0 * 365.25 * 86400.0)  # → m/s

# Best calibration head field
BEST_SCENARIO = str(ds.attrs.get('best_calibration', 'geology'))
head_key = f'head_{BEST_SCENARIO}'
if head_key not in ds:
    head_key = [k for k in ds.data_vars if k.startswith('head_')][0]
    warnings.warn(f'best_calibration key not found; using {head_key}')
head_current = ds[head_key].values.astype(float)     # current-climate head
hk           = ds['best_hk'].values.astype(float)    # hydraulic conductivity (m/s)

# ── Grid geometry ─────────────────────────────────────────────────────────────
nrow, ncol          = active.shape
cell_size_m         = float(ds.attrs['cell_size_m'])
delr = delc         = cell_size_m
aquifer_thickness_m = float(ds.attrs['aquifer_thickness_m'])
sea_level_base_m    = float(ds.attrs.get('sea_level_m', 0.0))

x_min  = float(ds.coords['x'].values[0]) - cell_size_m / 2.0
y_max  = float(ds.coords['y'].values[0]) + cell_size_m / 2.0
transform = Affine(cell_size_m, 0.0, x_min, 0.0, -cell_size_m, y_max)

sw_cells = (sw > 0) & (sw < 3)
is_sea   = sea.astype(bool)

grid = dict(
    dem=dem, active=active, sw_cells=sw_cells, is_sea=is_sea,
    nrow=nrow, ncol=ncol, delr=delr, delc=delc,
    transform=transform, aquifer_thickness_m=aquifer_thickness_m,
)

_S_PER_YR = 365.25 * 86400.0

print(f'Loaded scenario   : {head_key}')
print(f'Grid              : {nrow} rows × {ncol} cols  ({cell_size_m:.0f} m cells)')
print(f'Active cells      : {active.sum():,}')
print(f'Head range        : {np.nanmin(head_current[active]):.1f} – '
      f'{np.nanmax(head_current[active]):.1f} m a.s.l.')
print(f'Mean recharge     : {float(np.mean(rch[active])) * _S_PER_YR * 1000:.0f} mm/yr')


## Part B — Climate projections: *Klima i Norge 2100*

### Your task

1. Read the relevant chapter(s) of *Klima i Norge 2100* (Hanssen-Bauer et al.,
   2015; updated English edition: Hanssen-Bauer et al., 2017, MET Report 2/2017).
   The report is freely available from the Norwegian Climate Service Centre
   (klimaservicesenter.no).  Focus on **western Norway (Vestlandet)** and the
   RCP 8.5 scenario (high-end emissions) for the **2071–2100** time horizon.

2. Verify the indicative values in the table below against the report and **edit
   the Python variables** in the code cell to match your chosen scenario and region.
   Cite the report in your lab report.

### Reference table — indicative projections for western Norway

The values below are illustrative; the exact numbers depend on the sub-region,
emission scenario, and model ensemble used.  Check the report.

| Variable | RCP 4.5 (2071–2100) | RCP 8.5 (2071–2100) | Unit |
|---|---|---|---|
| Mean annual precipitation change | +8 to +12 | +15 to +20 | % |
| 1-day extreme precipitation (1-in-20-yr event) | +15 to +25 | +25 to +45 | % |
| Sea-level rise (RCP 4.5 / 8.5) | +0.3 to +0.5 | +0.5 to +0.9 | m |

*Sources: Hanssen-Bauer et al. (2015) Klima i Norge 2100, Miljødirektoratet M-405;
Hanssen-Bauer et al. (2017) Climate in Norway 2100, MET Report 2/2017;
Simpson et al. (2015) sea-level projections in Klima i Norge 2100.*

**Assumption:** recharge is proportional to precipitation (same percentage change).
This is a simplification — in reality, evapotranspiration and snowmelt also change —
but it is a standard first approximation used in Norwegian groundwater studies.

In [ ]:
# ── Student input: fill in values from Klima i Norge 2100 ─────────────────────
# Edit the numbers below to match your chosen region and scenario.
# The values here are illustrative RCP 8.5 / western Norway midpoint estimates.

delta_mean_precip_pct    = 18.0   # % change in mean annual precipitation    <-- verify
delta_extreme_precip_pct = 35.0   # % change in 1-day extreme precipitation  <-- verify
sea_level_rise_m         =  0.65  # sea-level rise (m)                       <-- verify

# Extreme rainfall depth for the instantaneous event (Part D)
# Typical 1-day / 1-in-20-year event for western Norway: 60–100 mm.
# Check senorge.no or eklima.met.no for your specific location.
P_extreme_current_m  = 0.080  # current-climate 1-day extreme rainfall (m)   <-- verify

# Effective porosity / specific yield for the shallow soil/aquifer
# Controls how much the water table rises per metre of rainfall.
# Typical values: sandy moraine 0.10–0.20; glaciomarine/silty soil 0.05–0.10
ne_storage = 0.12   # <-- change to match local geology

# ── Derived quantities ────────────────────────────────────────────────────────
recharge_multiplier_mean    = 1.0 + delta_mean_precip_pct    / 100.0
recharge_multiplier_extreme = 1.0 + delta_extreme_precip_pct / 100.0
future_sea_level_m          = sea_level_base_m + sea_level_rise_m

# Future extreme event rainfall = current × (1 + delta_extreme/100)
P_extreme_future_m = P_extreme_current_m * (1.0 + delta_extreme_precip_pct / 100.0)

print('Scenario parameters:')
print(f'  Δ mean precipitation         : +{delta_mean_precip_pct:.0f} %')
print(f'  Δ extreme precipitation      : +{delta_extreme_precip_pct:.0f} %')
print(f'  Sea-level rise               : +{sea_level_rise_m:.2f} m')
print(f'  Recharge multiplier (mean)   : {recharge_multiplier_mean:.3f}')
print(f'  Recharge multiplier (extreme): {recharge_multiplier_extreme:.3f}')
print(f'  Future sea level             : {future_sea_level_m:.2f} m a.s.l.')
print(f'  Extreme event (current)      : {P_extreme_current_m*1000:.0f} mm')
print(f'  Extreme event (future)       : {P_extreme_future_m*1000:.0f} mm')
print(f'  Effective porosity (storage) : {ne_storage}')


## Part C — Run climate scenarios

We run two additional steady-state simulations on top of the current-climate
baseline already loaded from Exercise 5B:

| Scenario | Recharge multiplier | Sea level |
|---|---|---|
| Current climate | 1.0 (calibrated) | 0 m |
| Future mean climate | 1 + Δ_mean/100 | +SLR |
| Future extreme climate | 1 + Δ_extreme/100 | +SLR |

The `recharge_multiplier` parameter in `gwu.simulate()` scales the entire
recharge raster uniformly — consistent with our proportionality assumption.

In [ ]:
# ── Scenario definitions ──────────────────────────────────────────────────────
scenario_defs = [
    dict(label='Future mean climate',
         recharge_multiplier=recharge_multiplier_mean,
         sea_level_m=future_sea_level_m),
    dict(label='Future extreme climate',
         recharge_multiplier=recharge_multiplier_extreme,
         sea_level_m=future_sea_level_m),
]

# ── Run new simulations ───────────────────────────────────────────────────────
head_scenarios = {'Current climate': head_current}

for sdef in scenario_defs:
    print(f"Running: {sdef['label']}  "
          f"(rch × {sdef['recharge_multiplier']:.3f}, "
          f"SL = {sdef['sea_level_m']:.2f} m) …")
    head_s, diag_s = gwu.simulate(
        hk_arr              = hk,
        dem                 = dem,
        sw                  = sw,
        sea                 = sea,
        active              = active,
        rch                 = rch,
        nrow                = nrow,
        ncol                = ncol,
        delr                = delr,
        delc                = delc,
        model_dir           = MODEL_DIR,
        aquifer_thickness_m = aquifer_thickness_m,
        recharge_multiplier = sdef['recharge_multiplier'],
        sea_level_m         = sdef['sea_level_m'],
    )
    head_scenarios[sdef['label']] = head_s
    print(f"  Done.  Head range: "
          f"{np.nanmin(head_s[active]):.1f} – {np.nanmax(head_s[active]):.1f} m a.s.l.")

print('\nAll scenario runs complete.')
print('Scenarios available:', list(head_scenarios.keys()))


## Part D — Instantaneous extreme-rainfall event

The steady-state model gives the long-term average water table for each climate
scenario.  We add a fourth scenario representing the **worst-case water table
immediately after a heavy rainfall event** by assuming that the rainfall fills the
available storage above the current water table instantaneously, with no time for
lateral drainage.

### The model

If the water table is currently at depth $d$ below the surface, a rainfall depth
$P$ (m) will raise it by:

$$\Delta d = \frac{P}{n_e}$$

where $n_e$ is the **specific yield** (effective porosity for storage/drainage).
The new depth after the event is:

$$d_{\text{event}} = \max\!\left(0,\; d_{\text{steady}} - \frac{P}{n_e}\right)$$

A depth of zero means the water table has reached (or exceeded) the surface —
i.e., the ground is fully saturated and standing water or surface runoff occurs.

### Limitations

This model is a deliberate simplification.  It assumes:
- All rainfall infiltrates instantly (no surface runoff, no interception).
- No lateral drainage occurs during the event.
- The aquifer is homogeneous with uniform $n_e$.

In reality, the response is governed by aquifer hydraulic diffusivity
($D = T / S$), preferential flow paths, and the duration of the event.  The
instantaneous model gives an **upper bound** on water-table rise — a conservative
estimate useful for hazard mapping.

In [ ]:
def _apply_extreme_event(head_steady, dem_, active_, P_m, ne):
    """
    Raise the water table by P/ne, capped at the land surface.

    Parameters
    ----------
    head_steady : ndarray  – steady-state head (m a.s.l.)
    dem_        : ndarray  – land-surface elevation (m a.s.l.)
    active_     : ndarray  – boolean active-cell mask
    P_m         : float    – rainfall depth (m)
    ne          : float    – specific yield / effective porosity

    Returns
    -------
    head_event : ndarray  – post-event head (m a.s.l.)
    """
    wt_depth   = np.where(active_, dem_ - head_steady, np.nan)   # depth below surface (m)
    rise       = P_m / ne                                          # WT rise from event (m)
    new_depth  = np.maximum(0.0, wt_depth - rise)                 # new depth (capped at 0)
    head_event = np.where(active_, dem_ - new_depth, np.nan)
    return head_event


# Apply to both current and future extreme steady states
head_current_event = _apply_extreme_event(
    head_scenarios['Current climate'], dem, active,
    P_extreme_current_m, ne_storage)

head_future_event  = _apply_extreme_event(
    head_scenarios['Future extreme climate'], dem, active,
    P_extreme_future_m, ne_storage)

# ── Collect all four head arrays with labels ──────────────────────────────────
all_scenarios = {
    'Current (steady state)'         : head_scenarios['Current climate'],
    'Future mean (steady state)'     : head_scenarios['Future mean climate'],
    'Future extreme (steady state)'  : head_scenarios['Future extreme climate'],
    'Future extreme + event'         : head_future_event,
}

# Quick summary: fraction of active land with WT within 0.5 m of surface
print('Water-table depth summary (active land cells only):')
print(f'  {"Scenario":<35}  {"median WT depth":>15}  {"% WT < 0.5 m":>14}')
print('  ' + '-' * 68)
for label, h in all_scenarios.items():
    wt_d = dem[active] - h[active]
    wt_d = wt_d[np.isfinite(wt_d)]
    med  = float(np.median(wt_d))
    pct  = 100.0 * float(np.mean(wt_d <= 0.5))
    print(f'  {label:<35}  {med:>14.2f} m  {pct:>13.1f} %')


## Part E — Groundwater flooding maps

We define **groundwater flooding** as a water-table depth ≤ `FLOOD_THRESHOLD_M`
below the land surface.  A threshold of 0.5 m is commonly used in Norwegian
groundwater flood risk assessments (NVE guidelines); at this depth, basement
floors, shallow foundations, and road sub-bases are at risk.

You can change `FLOOD_THRESHOLD_M` to explore sensitivity to the threshold choice.

In [ ]:
# ── Student input ─────────────────────────────────────────────────────────────
FLOOD_THRESHOLD_M = 0.5   # water-table depth threshold for flooding (m)  <-- edit

# ── Compute flooding masks and WT depth maps ──────────────────────────────────
wt_depths  = {}   # label → WT depth array (m below surface, NaN outside active)
flood_masks = {}  # label → bool raster

for label, h in all_scenarios.items():
    wt_d = np.where(active, dem - h, np.nan)
    wt_depths[label]   = wt_d
    # Exclude sea cells from flood analysis (always 'flooded' by definition)
    flood_masks[label] = active & ~is_sea & (wt_d <= FLOOD_THRESHOLD_M)

# ── 4-panel flooding maps ─────────────────────────────────────────────────────
ph = gwp._panel_h(nrow, ncol)
fig, axes = plt.subplots(2, 2, figsize=(12, 2 * ph),
                          squeeze=False)
fig.subplots_adjust(left=0.06, right=0.97, top=0.95, bottom=0.03,
                    hspace=0.45, wspace=0.38)

_SC_LABELS = list(all_scenarios.keys())
_SC_AXES   = [(0,0),(0,1),(1,0),(1,1)]
_PANEL_LETTERS = ['A','B','C','D']

# Common colour scale for WT depth (cmc.vik: blue=above surface, red=deep)
_wt_all = np.concatenate([wt_depths[l][active & ~is_sea & np.isfinite(wt_depths[l])]
                           for l in _SC_LABELS])
_wt_p2  = float(np.nanpercentile(_wt_all, 2))
_wt_p98 = float(np.nanpercentile(_wt_all, 98))
_wt_norm = TwoSlopeNorm(vmin=min(_wt_p2, -0.5), vcenter=0.0,
                         vmax=max(_wt_p98, 1.0))

for (ri, ci), label, letter in zip(_SC_AXES, _SC_LABELS, _PANEL_LETTERS):
    ax   = axes[ri][ci]
    wtd  = wt_depths[label]
    fmsk = flood_masks[label]
    h    = all_scenarios[label]

    # WT depth map
    im = ax.imshow(np.where(active & ~is_sea, wtd, np.nan),
                   cmap=cmc.vik, norm=_wt_norm)
    gwp._cbar(im, ax, 'WT depth (m)')

    # Flood overlay (orange hatch)
    flood_show = np.ma.masked_where(~fmsk, np.ones((nrow, ncol)))
    ax.imshow(flood_show, cmap='Oranges', vmin=0, vmax=1, alpha=0.7, zorder=5)

    n_flood = int(fmsk.sum())
    area_ha = n_flood * cell_size_m**2 / 1e4
    land_cells = int((active & ~is_sea).sum())
    pct_flood  = 100.0 * n_flood / max(land_cells, 1)

    ax.set_title(
        f'{letter}) {label}\n'
        f'Flooded (WT ≤ {FLOOD_THRESHOLD_M} m): {area_ha:.0f} ha  ({pct_flood:.1f} %)',
        fontsize=8)

    gwp.add_map_ticks(ax, grid)
    gwp.add_map_overlays(ax, grid)

# Legend
_patch_flood = mpatches.Patch(color='#f97316', alpha=0.7,
                               label=f'Flooded (WT ≤ {FLOOD_THRESHOLD_M} m)')
fig.legend(handles=[_patch_flood], loc='lower center', ncol=1,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
fig.suptitle(
    f'Groundwater flooding hazard — water-table depth (m below surface)\n'
    f'Blue = above/at surface  |  red = deep  |  orange = flooded zone (≤ {FLOOD_THRESHOLD_M} m)',
    fontsize=10)
plt.show()


In [ ]:
# ── Flooding summary table ────────────────────────────────────────────────────
land_cells = int((active & ~is_sea).sum())
flood_rows = []
for label in _SC_LABELS:
    fmsk   = flood_masks[label]
    wtd    = wt_depths[label]
    n_fld  = int(fmsk.sum())
    area   = n_fld * cell_size_m**2 / 1e4
    pct    = 100.0 * n_fld / max(land_cells, 1)
    valid  = wtd[active & ~is_sea & np.isfinite(wtd)]
    flood_rows.append(dict(
        Scenario          = label,
        Flooded_area_ha   = round(area, 1),
        Pct_flooded       = round(pct, 2),
        Median_WT_depth_m = round(float(np.median(valid)), 2),
        P10_WT_depth_m    = round(float(np.percentile(valid, 10)), 2),
    ))

df_flood = pd.DataFrame(flood_rows).set_index('Scenario')
df_flood.columns = ['Flooded area (ha)', '% flooded', 'Median WT depth (m)',
                     'P10 WT depth (m)']
print(f'Flood threshold: {FLOOD_THRESHOLD_M} m  |  '
      f'Total active land cells: {land_cells:,}  '
      f'({land_cells * cell_size_m**2 / 1e4:.0f} ha)\n')
display(df_flood.style
        .format({'Flooded area (ha)': '{:.1f}',
                 '% flooded': '{:.2f}',
                 'Median WT depth (m)': '{:.2f}',
                 'P10 WT depth (m)': '{:.2f}'})
        .background_gradient(subset=['Flooded area (ha)', '% flooded'],
                             cmap='Oranges')
       )


## Part F — Slope analysis and terrain classification

We compute terrain slope from the model-grid DEM using central finite differences.
Although the 100 m grid resolution smooths out sharp ridges and gullies that would
be visible on a 1–10 m lidar DEM, it is sufficient to identify the dominant
steep-terrain zones at the catchment scale.

The **resultant slope** magnitude (m/m) is:

$$s = \sqrt{\left(\frac{\partial z}{\partial x}\right)^2 + \left(\frac{\partial z}{\partial y}\right)^2}$$

and the **slope angle** in degrees is $\alpha = \arctan(s) \cdot 180/\pi$.

### Terrain classification used in hazard mapping

| Class | Slope angle | Implication |
|---|---|---|
| Flat / low | 0–5° | Minimal slope hazard; flood risk dominates |
| Moderate | 5–15° | Possible soil creep; some debris flow initiation |
| Steep | 15–25° | Elevated landslide and debris-flow susceptibility |
| Very steep | > 25° | High debris-flow and rockfall susceptibility |

In [ ]:
# ── Compute slope ─────────────────────────────────────────────────────────────
# np.gradient uses central differences on interior cells, one-sided at edges.
# dem values outside active domain are masked; fill with NaN before gradient
# then mask result back to active domain.
dem_masked = np.where(active, dem, np.nan)

# Fill NaN with nearest-neighbour for gradient (avoid large spurious gradients
# at the active boundary)
from scipy.ndimage import distance_transform_edt
def _fill_nan_nn(arr):
    """Fill NaN values with the nearest valid neighbour (for gradient calc)."""
    mask = ~np.isfinite(arr)
    if not mask.any():
        return arr.copy()
    idx  = distance_transform_edt(mask, return_distances=False, return_indices=True)
    filled = arr[tuple(idx)]
    return filled

dem_filled = _fill_nan_nn(dem_masked)

# Gradient in m/m (divide by cell size to convert pixel→physical units)
dz_dy, dz_dx = np.gradient(dem_filled, delr, delc)   # note: np.gradient(arr, row_step, col_step)
slope_mm  = np.sqrt(dz_dx**2 + dz_dy**2)             # resultant slope (m/m)
slope_deg = np.degrees(np.arctan(slope_mm))           # slope angle (degrees)

# Mask to active domain
slope_mm  = np.where(active, slope_mm,  np.nan)
slope_deg = np.where(active, slope_deg, np.nan)

# ── Terrain classification ────────────────────────────────────────────────────
_SLOPE_BREAKS = [0, 5, 15, 25, 90]   # degrees
_SLOPE_LABELS = ['Flat (0–5°)', 'Moderate (5–15°)', 'Steep (15–25°)', 'Very steep (>25°)']
_SLOPE_COLORS = ['#ffffcc', '#fdae61', '#d7191c', '#7b2d0a']

slope_class = np.full((nrow, ncol), -1, dtype=int)
for k, (lo, hi) in enumerate(zip(_SLOPE_BREAKS[:-1], _SLOPE_BREAKS[1:])):
    slope_class = np.where(active & (slope_deg >= lo) & (slope_deg < hi),
                           k, slope_class)

# ── Plot: slope angle + terrain class ────────────────────────────────────────
ph = gwp._panel_h(nrow, ncol)
fig, axes = plt.subplots(1, 2, figsize=(13, ph))
fig.subplots_adjust(left=0.06, right=0.97, top=0.94, bottom=0.06, wspace=0.42)

# Slope angle
_s_hi = float(np.nanpercentile(slope_deg[active], 98))
im0 = axes[0].imshow(slope_deg, cmap=cmc.lajolla, vmin=0, vmax=max(_s_hi, 5))
gwp._cbar(im0, axes[0], 'Slope angle (°)')
axes[0].set_title('A) Terrain slope (degrees)')
gwp.add_map_ticks(axes[0], grid)
gwp.add_map_overlays(axes[0], grid)

# Terrain class
_cmap_cls = mcolors.ListedColormap(_SLOPE_COLORS)
_bnorm    = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], _cmap_cls.N)
slope_cls_plot = np.where(active & (slope_class >= 0),
                           slope_class.astype(float), np.nan)
im1 = axes[1].imshow(slope_cls_plot, cmap=_cmap_cls, norm=_bnorm)
divider = make_axes_locatable(axes[1])
cax1 = divider.append_axes('right', size='3%', pad=0.08)
cb1  = plt.colorbar(im1, cax=cax1, ticks=[0,1,2,3])
cb1.set_ticklabels(_SLOPE_LABELS, fontsize=6)
axes[1].set_title('B) Terrain classification')
gwp.add_map_ticks(axes[1], grid)
gwp.add_map_overlays(axes[1], grid)

fig.suptitle('Slope analysis', fontsize=11)
plt.show()

# Summary
print('Terrain class distribution (active land cells):')
for k, lbl in enumerate(_SLOPE_LABELS):
    n = int(np.sum((slope_class == k) & active & ~is_sea))
    a = n * cell_size_m**2 / 1e4
    p = 100.0 * n / max(land_cells, 1)
    print(f'  {lbl:<25} : {n:6,} cells  ({a:7.0f} ha  /  {p:.1f} %)')
print(f'  Median slope (active domain): {np.nanmedian(slope_deg[active]):.1f}°')
print(f'  98th-pct slope              : {_s_hi:.1f}°')


## Part G — Combined hazard index and maps

We combine the modelled water-table depth with terrain slope to identify two
types of hazard zone:

### 1. Groundwater flood hazard
Water table ≤ `FLOOD_THRESHOLD_M` **AND** slope < 15° (flat to moderate terrain
where standing/ponded groundwater can accumulate).

### 2. Rainfall-triggered landslide / debris-flow hazard
Water table ≤ `SLIDE_WT_THRESHOLD_M` **AND** slope ≥ 15° (saturated or
near-saturated conditions on steep slopes).  The water-table threshold for
landslide hazard can be set more conservatively (deeper than for flooding) because
even partial saturation increases pore pressure enough to reduce slope stability.

### TOPMODEL-style wetness index (for reference)
The classic TOPMODEL wetness index is:

$$\text{TWI} = \ln\!\left(\frac{a}{\tan\alpha}\right)$$

where $a$ is the upslope contributing area per unit contour length and $\alpha$ is
the slope angle.  Here we replace $a$ with the inverse of the modelled water-table
depth, giving a modified index:

$$\text{TWI}_{\text{mod}} = \ln\!\left(\frac{1}{d_{\text{WT}} \cdot \tan\alpha + \varepsilon}\right)$$

High TWI$_{\text{mod}}$ values indicate cells that are likely to be wet
regardless of upslope area — driven directly by the modelled groundwater state.

In [ ]:
# ── Student input ─────────────────────────────────────────────────────────────
SLIDE_WT_THRESHOLD_M = 1.0   # WT depth threshold for landslide hazard (m)  <-- edit
SLIDE_SLOPE_MIN_DEG  = 15.0  # minimum slope for landslide hazard (°)       <-- edit
FLOOD_SLOPE_MAX_DEG  = 15.0  # maximum slope for flooding hazard (°)        <-- edit

# ── Hazard class definitions ──────────────────────────────────────────────────
# 0 = No elevated hazard
# 1 = Flood hazard only
# 2 = Landslide/debris-flow hazard only
# 3 = Both hazards
_HAZ_COLORS = ['#d9d9d9', '#4575b4', '#d73027', '#762a83']
_HAZ_LABELS = ['No elevated hazard', 'Flood hazard',
               'Landslide hazard', 'Both hazards']

hazard_maps = {}

def _compute_hazard(wt_depth, slope_deg_, active_, is_sea_,
                    flood_thr, slide_thr, slide_slope_min, flood_slope_max):
    """Return integer hazard class raster (0–3)."""
    land = active_ & ~is_sea_ & np.isfinite(wt_depth)
    flood_zone = land & (wt_depth <= flood_thr) & (slope_deg_ < flood_slope_max)
    slide_zone = land & (wt_depth <= slide_thr) & (slope_deg_ >= slide_slope_min)
    haz = np.zeros((wt_depth.shape), dtype=int)
    haz[flood_zone] += 1
    haz[slide_zone] += 2
    haz = np.where(land, haz, -1)   # -1 = inactive/sea
    return haz

for label in _SC_LABELS:
    haz = _compute_hazard(
        wt_depths[label], slope_deg, active, is_sea,
        FLOOD_THRESHOLD_M, SLIDE_WT_THRESHOLD_M,
        SLIDE_SLOPE_MIN_DEG, FLOOD_SLOPE_MAX_DEG)
    hazard_maps[label] = haz

# ── Modified TOPMODEL wetness index (future extreme + event) ─────────────────
_eps    = 1e-3
_tan_sl = np.where(active, np.tan(np.radians(np.maximum(slope_deg, 0.1))), np.nan)
_wtd_fe = np.where(active, np.maximum(wt_depths['Future extreme + event'], _eps), np.nan)
TWI_mod = np.where(active, np.log(1.0 / (_wtd_fe * _tan_sl + _eps)), np.nan)

# ── 4-panel hazard maps ───────────────────────────────────────────────────────
ph = gwp._panel_h(nrow, ncol)
fig, axes = plt.subplots(2, 2, figsize=(12, 2 * ph), squeeze=False)
fig.subplots_adjust(left=0.06, right=0.97, top=0.94, bottom=0.05,
                    hspace=0.48, wspace=0.38)

_cmap_haz = mcolors.ListedColormap(_HAZ_COLORS)
_bnorm_haz = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], _cmap_haz.N)

for (ri, ci), label, letter in zip(_SC_AXES, _SC_LABELS, _PANEL_LETTERS):
    ax  = axes[ri][ci]
    haz = hazard_maps[label]
    haz_plot = np.where(haz >= 0, haz.astype(float), np.nan)

    im = ax.imshow(haz_plot, cmap=_cmap_haz, norm=_bnorm_haz)

    n_tot  = int(np.sum(haz >= 0))
    counts = [int(np.sum(haz == k)) for k in range(4)]
    areas  = [c * cell_size_m**2 / 1e4 for c in counts]

    ax.set_title(
        f'{letter}) {label}\n'
        f'Flood: {areas[1]:.0f} ha  |  Slide: {areas[2]:.0f} ha  |  '
        f'Both: {areas[3]:.0f} ha',
        fontsize=7.5)

    gwp.add_map_ticks(ax, grid)
    gwp.add_map_overlays(ax, grid)

# Shared colorbar
_cax_haz = fig.add_axes([0.15, 0.01, 0.70, 0.014])
_cb_haz  = plt.colorbar(im, cax=_cax_haz, orientation='horizontal',
                         ticks=[0,1,2,3])
_cb_haz.set_ticklabels(_HAZ_LABELS, fontsize=8)

fig.suptitle(
    f'Combined hazard index\n'
    f'Flood: WT ≤ {FLOOD_THRESHOLD_M} m & slope < {FLOOD_SLOPE_MAX_DEG}°  |  '
    f'Landslide: WT ≤ {SLIDE_WT_THRESHOLD_M} m & slope ≥ {SLIDE_SLOPE_MIN_DEG}°',
    fontsize=9)
plt.show()


In [ ]:
# ── Modified TWI map (future extreme + event) ─────────────────────────────────
ph = gwp._panel_h(nrow, ncol)
fig, ax = plt.subplots(figsize=(7, ph))
_twi_lo = float(np.nanpercentile(TWI_mod[active], 2))
_twi_hi = float(np.nanpercentile(TWI_mod[active], 98))
im_twi  = ax.imshow(np.where(active, TWI_mod, np.nan),
                    cmap=cmc.hawaii, vmin=_twi_lo, vmax=_twi_hi)
gwp._cbar(im_twi, ax, 'TWI_mod (log scale)')
ax.set_title(
    'Modified TOPMODEL wetness index — Future extreme + event\n'
    'High values = shallow WT on low slope (flood) or steep slope (landslide)')
gwp.add_map_ticks(ax, grid)
gwp.add_map_overlays(ax, grid)
plt.show()


## Part H — Scenario comparison, change maps, and reflection

### Change maps

How much does the water table rise between the current climate and future
scenarios?  A **change map** (Δhead = head_future − head_current) makes the
spatial pattern of change immediately visible.

In [ ]:
# ── ΔHead change maps ─────────────────────────────────────────────────────────
_change_pairs = [
    ('Future mean (steady state)',    'A) Δ Head: future mean − current'),
    ('Future extreme (steady state)', 'B) Δ Head: future extreme − current'),
    ('Future extreme + event',        'C) Δ Head: future extreme + event − current'),
]

ph = gwp._panel_h(nrow, ncol)
fig, axes = plt.subplots(1, 3, figsize=(18, ph))
fig.subplots_adjust(left=0.05, right=0.98, top=0.93, bottom=0.06, wspace=0.42)

h_ref = all_scenarios['Current (steady state)']

for ax, (fut_label, title) in zip(axes, _change_pairs):
    dh = np.where(active, all_scenarios[fut_label] - h_ref, np.nan)
    _dh_abs_max = max(float(np.nanpercentile(np.abs(dh[active]), 99)), 0.1)
    _dh_norm    = TwoSlopeNorm(vmin=-_dh_abs_max, vcenter=0.0, vmax=_dh_abs_max)
    im = ax.imshow(dh, cmap=cmc.vik, norm=_dh_norm)
    gwp._cbar(im, ax, 'Δ Head (m)')
    med_rise = float(np.nanmedian(dh[active]))
    p90_rise = float(np.nanpercentile(dh[active], 90))
    ax.set_title(f'{title}\nMedian: +{med_rise:.2f} m  |  P90: +{p90_rise:.2f} m',
                 fontsize=8)
    gwp.add_map_ticks(ax, grid)
    gwp.add_map_overlays(ax, grid)

fig.suptitle('Water-table change relative to current climate (blue = rise, red = fall)',
             fontsize=10)
plt.show()


In [ ]:
# ── Hazard area summary table ─────────────────────────────────────────────────
haz_rows = []
for label in _SC_LABELS:
    haz    = hazard_maps[label]
    counts = [int(np.sum(haz == k)) for k in range(4)]
    areas  = [c * cell_size_m**2 / 1e4 for c in counts]
    pcts   = [100.0 * c / max(land_cells, 1) for c in counts]
    haz_rows.append(dict(
        Scenario         = label,
        No_hazard_ha     = areas[0],
        Flood_ha         = areas[1],
        Landslide_ha     = areas[2],
        Both_ha          = areas[3],
        Pct_flood        = pcts[1],
        Pct_landslide    = pcts[2],
        Pct_both         = pcts[3],
        Pct_any_hazard   = pcts[1] + pcts[2] + pcts[3],
    ))

df_haz = pd.DataFrame(haz_rows).set_index('Scenario')

print('Hazard area summary (active land, excluding sea):')
display(df_haz[['Flood_ha','Landslide_ha','Both_ha',
                'Pct_flood','Pct_landslide','Pct_both','Pct_any_hazard']]
        .rename(columns={
            'Flood_ha': 'Flood (ha)',
            'Landslide_ha': 'Landslide (ha)',
            'Both_ha': 'Both (ha)',
            'Pct_flood': 'Flood (%)',
            'Pct_landslide': 'Slide (%)',
            'Pct_both': 'Both (%)',
            'Pct_any_hazard': 'Any hazard (%)'
        })
        .style.format('{:.1f}')
        .background_gradient(subset=['Flood (%)','Slide (%)','Both (%)'],
                             cmap='YlOrRd', axis=None)
       )


In [ ]:
# ── Bar chart: hazard area by scenario ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.subplots_adjust(left=0.08, right=0.97, top=0.90, bottom=0.20, wspace=0.42)

x = np.arange(len(_SC_LABELS))
width = 0.28

for ax, (cols, title, unit) in zip(axes, [
    (['Flood (ha)', 'Landslide (ha)', 'Both (ha)'],
     'A) Hazard area by scenario', 'Area (ha)'),
    (['Flood (%)', 'Slide (%)', 'Both (%)'],
     'B) Hazard area as % of active land', '% of active land'),
]):
    df_plot = df_haz.rename(columns={
        'Flood_ha': 'Flood (ha)', 'Landslide_ha': 'Landslide (ha)', 'Both_ha': 'Both (ha)',
        'Pct_flood': 'Flood (%)', 'Pct_landslide': 'Slide (%)', 'Pct_both': 'Both (%)'
    })
    for i, (col, color) in enumerate(zip(cols, ['#4575b4', '#d73027', '#762a83'])):
        vals = df_plot[col].values
        bars = ax.bar(x + (i - 1) * width, vals, width,
                      color=color, alpha=0.85, label=col, edgecolor='white', linewidth=0.6)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(vals.max()*0.01, 0.1),
                    f'{v:.0f}', ha='center', va='bottom', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(_SC_LABELS, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel(unit)
    ax.set_title(title)
    ax.legend(fontsize=8, framealpha=0.85)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Hazard area comparison across climate scenarios', fontsize=11)
plt.show()


## Part H — Reflection questions

Use the maps and tables from Parts E–H to answer the following questions in your
report (~0.5–1 page).  Integrate your answers into a coherent discussion rather
than a list of numbered replies.

### Questions to address

1. **Location of flooding hazard.**  
   Where are the areas most susceptible to groundwater flooding in the current
   climate?  What geological and topographic factors control the spatial pattern?
   Are these areas already identified as flood-risk zones in Norwegian planning
   maps (e.g., NVE's flomsonekart or Hva skjer i mitt nærområde)?

2. **Climate change signal.**  
   How much does the flooded area change between the current climate and the
   future mean and extreme scenarios?  Is the response roughly linear with the
   recharge multiplier?  Where does the largest change occur, and can you explain
   why those locations are most sensitive?

3. **Extreme event vs. steady-state.**  
   Compare the 'Future extreme (steady state)' and 'Future extreme + event'
   hazard maps.  How much additional flooding/landslide risk does the
   instantaneous event add on top of the already-raised steady-state water table?
   What does this imply about the relative importance of long-term recharge
   trends vs. individual storm events?

4. **Landslide and debris-flow hazard.**  
   Describe the areas flagged as elevated landslide hazard.  Are they associated
   with specific geological units (check the geology map from Exercise 5A/5B),
   valley sides, or coastal scarps?  How does the hazard extent change under
   future climate?

5. **Co-location of hazards.**  
   Are there areas flagged for *both* flooding and landslide hazard (purple in
   the combined hazard map)?  Explain physically why both hazards can co-occur
   in the same cell at this model resolution.

6. **Sea-level rise contribution.**  
   The future scenarios include both increased recharge and sea-level rise.  How
   would you isolate the contribution of sea-level rise alone?  In which parts of
   the catchment would you expect sea-level rise to dominate over recharge
   increase?

7. **Limitations and uncertainty.**  
   List at least three significant simplifications in this analysis.  Consider:
   the proportionality assumption for recharge, the instantaneous event model,
   the 100 m DEM resolution for slope, the steady-state solver, and the
   single-layer aquifer assumption.  How would each simplification tend to
   over- or under-estimate the hazard?

8. **Planning implications.**  
   Identify one specific measure from Norwegian climate adaptation practice
   (e.g., *Klimatilpasning Norge*, NVE guidelines, TEK17) that could reduce each
   of the two hazard types you have mapped.  Refer to specific map areas in your
   answer.

---

### Space for your written answer

*Write your answer in the Markdown cell below.  Aim for 0.5–1 page of continuous
text, not a bullet list.*

*(Your answer here)*